In [1]:
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, sys, re

In [2]:
adata = ad.read_h5ad('/work/magroup/skrieger/Syndri/data/datasets/agingbrain_merfish/raw/single-cell/752ed29d-bbb0-45bd-9c57-19edf68ef73e.h5ad')
st_data = ad.read_h5ad('/work/magroup/skrieger/Syndri/data/datasets/agingbrain_merfish/raw/spatial/c93d78c2-ee17-4504-8d1c-17cf093ad7b5.h5ad')
# Preserve the spatial panel's gene order when selecting shared genes.
gene_overlap = st_data.var_names.intersection(adata.var_names, sort=False)
st_data = st_data[:, gene_overlap].copy()
adata = adata[:, gene_overlap].copy()

In [3]:
# Restore raw counts by gene ID. AnnData.raw retains its original gene order
# after slicing var, so assigning raw.X directly would mislabel spatial counts.
if st_data.raw is None or adata.raw is None:
    raise ValueError('Both input AnnData objects must have raw counts in .raw')
st_data.X = st_data.raw[:, st_data.var_names].X.copy()
adata.X = adata.raw[:, adata.var_names].X.copy()
assert st_data.var_names.equals(adata.var_names)

In [4]:
st_data.obs['cond_label'] = st_data.obs['donor_id'].astype(str) + "_" + st_data.obs['slice'].astype(str)
st_data.obs['cond_label'].unique()

array(['MsBrainAgingSpatialDonor_1_0', 'MsBrainAgingSpatialDonor_2_1',
       'MsBrainAgingSpatialDonor_2_0', 'MsBrainAgingSpatialDonor_3_1',
       'MsBrainAgingSpatialDonor_3_0', 'MsBrainAgingSpatialDonor_4_1',
       'MsBrainAgingSpatialDonor_4_2', 'MsBrainAgingSpatialDonor_4_0',
       'MsBrainAgingSpatialDonor_5_0', 'MsBrainAgingSpatialDonor_5_2',
       'MsBrainAgingSpatialDonor_5_1', 'MsBrainAgingSpatialDonor_6_0',
       'MsBrainAgingSpatialDonor_6_1', 'MsBrainAgingSpatialDonor_6_2',
       'MsBrainAgingSpatialDonor_7_2', 'MsBrainAgingSpatialDonor_7_1',
       'MsBrainAgingSpatialDonor_7_0', 'MsBrainAgingSpatialDonor_8_0',
       'MsBrainAgingSpatialDonor_8_1', 'MsBrainAgingSpatialDonor_8_2',
       'MsBrainAgingSpatialDonor_9_2', 'MsBrainAgingSpatialDonor_9_1',
       'MsBrainAgingSpatialDonor_9_0', 'MsBrainAgingSpatialDonor_10_0',
       'MsBrainAgingSpatialDonor_10_1', 'MsBrainAgingSpatialDonor_10_2',
       'MsBrainAgingSpatialDonor_11_0', 'MsBrainAgingSpatialDonor_11_1',
 

In [5]:
st_data = st_data[st_data.obs['donor_id'].str.contains('Donor_5|Donor_6|Donor_7|Donor_8')]
st_data.obs['donor_id'] = "10X_"+st_data.obs['donor_id'].astype(str)

/tmp/ipykernel_935187/1657894079.py:2: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  st_data.obs['donor_id'] = "10X_"+st_data.obs['donor_id'].astype(str)


In [6]:
adata.obs['donor_id'],st_data.obs['donor_id']

(AAACCCAAGAACGCGT-1      10X_MsBrainAgingSpatialDonor_5
 AAACCCAAGAGCAACC-1      10X_MsBrainAgingSpatialDonor_5
 AAACCCAAGCATATGA-1      10X_MsBrainAgingSpatialDonor_5
 AAACCCAAGTGATTCC-1      10X_MsBrainAgingSpatialDonor_5
 AAACCCACAACAGCCC-1      10X_MsBrainAgingSpatialDonor_5
                                      ...              
 TTTGTTGTCCCGAAAT-1      10X_MsBrainAgingSpatialDonor_8
 TTTGTTGTCCGATTAG-1      10X_MsBrainAgingSpatialDonor_8
 TTTGTTGTCCTCTAAT-1-1    10X_MsBrainAgingSpatialDonor_8
 TTTGTTGTCTCTCTAA-1      10X_MsBrainAgingSpatialDonor_8
 TTTGTTGTCTGAGGCC-1      10X_MsBrainAgingSpatialDonor_8
 Name: donor_id, Length: 79667, dtype: category
 Categories (4, object): ['10X_MsBrainAgingSpatialDonor_5', '10X_MsBrainAgingSpatialDonor_6', '10X_MsBrainAgingSpatialDonor_7', '10X_MsBrainAgingSpatialDonor_8'],
 334480434318187743827450597969066295150    10X_MsBrainAgingSpatialDonor_5
 168543327967641209670519985000962475574    10X_MsBrainAgingSpatialDonor_5
 7063323882150196363402

In [7]:
import sys
import importlib
import numpy as np

sys.path.append("/home/jwinchel/Syndri/code/")
import mapping
importlib.reload(mapping)
from mapping import integrate_single_st_file_with_harmony

slice_col = "cond_label"
donor_col = "donor_id"
n_pcs = 30

reference_source = adata
reference_integrated = adata.copy()

if not reference_source.obs_names.is_unique:
    raise ValueError("Reference .obs_names must be unique.")

slice_labels = st_data.obs[slice_col].astype("string")
reference_donors = reference_source.obs[donor_col].astype("string")
slices = sorted(slice_labels.dropna().unique())
print(slices)
st_data_list = []
embedding_keys = {}
slice_donors = {}

for sl in slices:
    slice_mask = slice_labels.eq(sl).fillna(False).to_numpy(dtype=bool)
    spatial_subset = st_data[slice_mask].copy()

    donors = spatial_subset.obs[donor_col].astype("string")
    if donors.isna().any() or donors.nunique() != 1:
        raise ValueError(f"Slice {sl!r} must have exactly one nonmissing donor_id.")

    donor = str(donors.iloc[0])
    ref_mask = reference_donors.eq(donor).fillna(False).to_numpy(dtype=bool)

    if not ref_mask.any():
        raise ValueError(f"No reference cells for donor {donor!r}.")

    # Slices from the same donor use the same reference cells,
    # but each slice gets its own independent Harmony integration.
    ref_subset, sl_adata, key = integrate_single_st_file_with_harmony(
        spatial_subset,
        reference_source[ref_mask].copy(),
        tissue_filter=None,
        n_pcs=n_pcs,
        tissue_name=str(sl),
    )

    if key in embedding_keys.values():
        raise ValueError(f"Slice names produce a duplicate embedding key: {key}")

    subset_embedding = np.asarray(ref_subset.obsm[key])
    full_embedding = np.full(
        (reference_source.n_obs, subset_embedding.shape[1]),
        np.nan,
        dtype=subset_embedding.dtype,
    )

    positions = reference_source.obs_names.get_indexer(ref_subset.obs_names)
    if (positions < 0).any():
        raise ValueError("Integrated reference contains unexpected cell IDs.")

    full_embedding[positions] = subset_embedding
    reference_integrated.obsm[key] = full_embedding

    embedding_keys[str(sl)] = key
    slice_donors[str(sl)] = donor
    st_data_list.append(sl_adata)

reference_integrated.uns["slice_harmony"] = {
    "slice_col": slice_col,
    "donor_col": donor_col,
    "slice_donors": slice_donors,
    "embedding_keys": embedding_keys,
}
adata = reference_integrated

# For downstream matching, select the eligible donor's reference rows:
# key = embedding_keys[str(sl)]
# eligible = np.isfinite(adata.obsm[key]).all(axis=1)
# reference_for_slice = adata[eligible].copy()

['MsBrainAgingSpatialDonor_5_0', 'MsBrainAgingSpatialDonor_5_1', 'MsBrainAgingSpatialDonor_5_2', 'MsBrainAgingSpatialDonor_6_0', 'MsBrainAgingSpatialDonor_6_1', 'MsBrainAgingSpatialDonor_6_2', 'MsBrainAgingSpatialDonor_7_0', 'MsBrainAgingSpatialDonor_7_1', 'MsBrainAgingSpatialDonor_7_2', 'MsBrainAgingSpatialDonor_8_0', 'MsBrainAgingSpatialDonor_8_1', 'MsBrainAgingSpatialDonor_8_2']


/home/jwinchel/Syndri/code/mapping.py:65: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata = sc_data.concatenate(
/home/jwinchel/miniconda3/envs/rctd/lib/python3.10/functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)
2026-10-10 21:09:14,551 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...


['age', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'n_genes', 'n_counts', 'clust_annot', 'sex_ontology_term_id', 'suspension_type', 'cell_type_ontology_term_id', 'assay_ontology_term_id', 'tissue_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'donor_id', 'is_primary_data', 'cell_type_annot', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'batch', 'fov', 'center_x', 'center_y', 'min_x', 'max_x', 'min_y', 'max_y', 'slice', 'cond_label']
category
batch
sc    21728
st    10744
Name: count, dtype: int64
(32472, 30)


2026-10-10 21:09:17,342 - harmonypy - INFO - sklearn.KMeans initialization complete.
2026-10-10 21:09:17,393 - harmonypy - INFO - Iteration 1 of 30
2026-10-10 21:09:21,465 - harmonypy - INFO - Iteration 2 of 30
2026-10-10 21:09:24,414 - harmonypy - INFO - Converged after 2 iterations
/home/jwinchel/Syndri/code/mapping.py:65: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata = sc_data.concatenate(
/home/jwinchel/miniconda3/envs/rctd/lib/python3.10/functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)
2026-10-10 21:09:26,487 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...


['age', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'n_genes', 'n_counts', 'clust_annot', 'sex_ontology_term_id', 'suspension_type', 'cell_type_ontology_term_id', 'assay_ontology_term_id', 'tissue_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'donor_id', 'is_primary_data', 'cell_type_annot', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'batch', 'fov', 'center_x', 'center_y', 'min_x', 'max_x', 'min_y', 'max_y', 'slice', 'cond_label']
category
batch
sc    21728
st     9644
Name: count, dtype: int64
(31372, 30)


2026-10-10 21:09:29,177 - harmonypy - INFO - sklearn.KMeans initialization complete.
2026-10-10 21:09:29,225 - harmonypy - INFO - Iteration 1 of 30
2026-10-10 21:09:32,159 - harmonypy - INFO - Iteration 2 of 30
2026-10-10 21:09:35,213 - harmonypy - INFO - Converged after 2 iterations
/home/jwinchel/Syndri/code/mapping.py:65: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata = sc_data.concatenate(
/home/jwinchel/miniconda3/envs/rctd/lib/python3.10/functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)
2026-10-10 21:09:37,027 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...


... (16 outputs left out)

['age', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'n_genes', 'n_counts', 'clust_annot', 'sex_ontology_term_id', 'suspension_type', 'cell_type_ontology_term_id', 'assay_ontology_term_id', 'tissue_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'donor_id', 'is_primary_data', 'cell_type_annot', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'batch', 'fov', 'center_x', 'center_y', 'min_x', 'max_x', 'min_y', 'max_y', 'slice', 'cond_label']
category
batch
sc    18543
st    12180
Name: count, dtype: int64
(30723, 30)


2026-10-10 21:11:38,817 - harmonypy - INFO - sklearn.KMeans initialization complete.
2026-10-10 21:11:38,865 - harmonypy - INFO - Iteration 1 of 30
2026-10-10 21:11:41,782 - harmonypy - INFO - Iteration 2 of 30
2026-10-10 21:11:44,925 - harmonypy - INFO - Iteration 3 of 30
2026-10-10 21:11:48,023 - harmonypy - INFO - Converged after 3 iterations
/home/jwinchel/Syndri/code/mapping.py:65: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata = sc_data.concatenate(
/home/jwinchel/miniconda3/envs/rctd/lib/python3.10/functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)
2026-10-10 21:11:50,189 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...


['age', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'n_genes', 'n_counts', 'clust_annot', 'sex_ontology_term_id', 'suspension_type', 'cell_type_ontology_term_id', 'assay_ontology_term_id', 'tissue_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'donor_id', 'is_primary_data', 'cell_type_annot', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'batch', 'fov', 'center_x', 'center_y', 'min_x', 'max_x', 'min_y', 'max_y', 'slice', 'cond_label']
category
batch
sc    18543
st    12036
Name: count, dtype: int64
(30579, 30)


2026-10-10 21:11:52,801 - harmonypy - INFO - sklearn.KMeans initialization complete.
2026-10-10 21:11:52,849 - harmonypy - INFO - Iteration 1 of 30
2026-10-10 21:11:55,676 - harmonypy - INFO - Iteration 2 of 30
2026-10-10 21:11:58,587 - harmonypy - INFO - Iteration 3 of 30
2026-10-10 21:12:01,477 - harmonypy - INFO - Converged after 3 iterations


In [8]:
adata.write_h5ad('/work/magroup/skrieger/Syndri/data/datasets/agingbrain_merfish/interim/single-cell/sc_integrated_agematched.h5ad')

In [9]:
slices = sorted(list(st_data.obs['cond_label'].unique()))
combined = np.full((st_data.n_obs,30),np.nan,dtype=adata.obsm['X_pca_harmony_MsBrainAgingSpatialDonor_5_0'].dtype)
# del adata
# map full adata cell IDs -> row indices
obs_to_idx = {
    obs_name: i
    for i, obs_name in enumerate(st_data.obs_names)
}

for i,subset in enumerate(st_data_list):
    idx = np.array([
        obs_to_idx[obs_name]
        for obs_name in subset.obs_names
    ])
    label = subset.obs['cond_label'].unique().tolist()[0]

    combined[idx] = subset.obsm[f"X_pca_harmony_{label}"]

st_data.obsm['X_pca_harmony'] = combined
st_data
# del st_data_list

AnnData object with n_obs × n_vars = 126408 × 374
    obs: 'fov', 'center_x', 'center_y', 'min_x', 'max_x', 'min_y', 'max_y', 'age', 'clust_annot', 'slice', 'sex_ontology_term_id', 'suspension_type', 'cell_type_ontology_term_id', 'assay_ontology_term_id', 'tissue_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'donor_id', 'is_primary_data', 'cell_type_annot', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'cond_label'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    uns: 'batch_condition', 'citation', 'is_pre_analysis', 'organism', 'organism_ontology_term_id', 'schema_reference', 'schema_version', 'title'
    obsm: 'X_pca', 'X_spatial_coords', 'X_umap', 'spatial_coords', 'X_pca_harmony'

In [10]:
for sl in slices:
    st_data[st_data.obs['cond_label']==sl].write_h5ad(f'/work/magroup/skrieger/Syndri/data/datasets/agingbrain_merfish/interim/spatial/st_integrated_{sl}.h5ad')

/home/jwinchel/miniconda3/envs/rctd/lib/python3.10/site-packages/anndata/_core/anndata.py:1138: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c
/home/jwinchel/miniconda3/envs/rctd/lib/python3.10/site-packages/anndata/_core/anndata.py:1138: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c
/home/jwinchel/miniconda3/envs/rctd/lib/python3.10/site-packages/anndata/_core/anndata.py:1138: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c
/home/jwinchel/miniconda3/envs/rctd/lib/python3.10/site-packages/anndata/_core/anndata.py:1138: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c
/home/jwinchel/miniconda3/envs/rctd/lib/python3.10/site-packages/anndata/_core/anndata.py:1138: ImplicitModificationWarning: Trying to modify attribute `.ob

In [11]:
adata

AnnData object with n_obs × n_vars = 79667 × 374
    obs: 'age', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'n_genes', 'n_counts', 'clust_annot', 'sex_ontology_term_id', 'suspension_type', 'cell_type_ontology_term_id', 'assay_ontology_term_id', 'tissue_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'donor_id', 'is_primary_data', 'cell_type_annot', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'feature_is_filtered', 'mean', 'std', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    uns: 'age_colors', 'batch_condition', 'citation', 'is_pre_analysis', 'leiden', 'neighbors', 'organism', 'organism_ontology_term_id', 'pca', 'schema_reference', 'schema_version', 'title', 'umap', 'slice_harmo

In [12]:
# read in previously saved single-cell and spatial data
# adata=ad.read_h5ad('/work/magroup/skrieger/Syndri/data/datasets/agingbrain_merfish/interim/single-cell/sc_integrated.h5ad')
# st_data = ad.read_h5ad('/work/magroup/skrieger/Syndri/data/datasets/agingbrain_merfish/interim/spatial/st_integrated_slice0.h5ad')
st_data.obs[['center_x','center_y']] = st_data.obs[['center_x','center_y']].astype(float)
st_data.obs[['min_x','min_y']] = st_data.obs[['min_x','min_y']].astype(float)
# st_data = st_data[st_data.obs['age']=='90wk']
# adata.obsm['X_pca_harmony'] = adata.obsm['X_pca_harmony_0'].copy()

In [13]:
adata.uns['slice_harmony']

{'slice_col': 'cond_label',
 'donor_col': 'donor_id',
 'slice_donors': {'MsBrainAgingSpatialDonor_5_0': '10X_MsBrainAgingSpatialDonor_5',
  'MsBrainAgingSpatialDonor_5_1': '10X_MsBrainAgingSpatialDonor_5',
  'MsBrainAgingSpatialDonor_5_2': '10X_MsBrainAgingSpatialDonor_5',
  'MsBrainAgingSpatialDonor_6_0': '10X_MsBrainAgingSpatialDonor_6',
  'MsBrainAgingSpatialDonor_6_1': '10X_MsBrainAgingSpatialDonor_6',
  'MsBrainAgingSpatialDonor_6_2': '10X_MsBrainAgingSpatialDonor_6',
  'MsBrainAgingSpatialDonor_7_0': '10X_MsBrainAgingSpatialDonor_7',
  'MsBrainAgingSpatialDonor_7_1': '10X_MsBrainAgingSpatialDonor_7',
  'MsBrainAgingSpatialDonor_7_2': '10X_MsBrainAgingSpatialDonor_7',
  'MsBrainAgingSpatialDonor_8_0': '10X_MsBrainAgingSpatialDonor_8',
  'MsBrainAgingSpatialDonor_8_1': '10X_MsBrainAgingSpatialDonor_8',
  'MsBrainAgingSpatialDonor_8_2': '10X_MsBrainAgingSpatialDonor_8'},
 'embedding_keys': {'MsBrainAgingSpatialDonor_5_0': 'X_pca_harmony_MsBrainAgingSpatialDonor_5_0',
  'MsBrainAging

In [14]:
import sys
import numpy as np
import anndata as ad

sys.path.append("/home/jwinchel/Syndri/code")
from mapping import map_sc_reference_to_unique_xenium_cells

slice_col = "cond_label"
donor_col = "donor_id"
embedding_keys = adata.uns["slice_harmony"]["embedding_keys"]

reference_donors = adata.obs[donor_col].astype("string")
rng = np.random.default_rng(0)

mapped_slices = []
slice_names = []
mapping_metadata = {}

for sl_adata in st_data_list:
    slices = sl_adata.obs[slice_col].astype("string")
    donors = sl_adata.obs[donor_col].astype("string")

    if slices.isna().any() or slices.nunique() != 1:
        raise ValueError("Each spatial AnnData must contain exactly one slice.")
    if donors.isna().any() or donors.nunique() != 1:
        raise ValueError("Each spatial slice must contain exactly one donor.")
    if not sl_adata.obs_names.is_unique:
        raise ValueError("Spatial .obs_names must be unique within each slice.")

    sl = str(slices.iloc[0])
    donor = str(donors.iloc[0])
    key = embedding_keys[sl]

    # Restrict reference candidates to this donor and this slice's embedding.
    donor_mask = reference_donors.eq(donor).fillna(False).to_numpy(dtype=bool)
    finite_mask = np.isfinite(np.asarray(adata.obsm[key])).all(axis=1)
    sc_subset = adata[donor_mask & finite_mask].copy()

    if sc_subset.n_obs == 0:
        raise ValueError(f"No eligible reference cells for slice {sl!r}.")
    if not np.isfinite(np.asarray(sl_adata.obsm[key])).all():
        raise ValueError(f"Nonfinite spatial embedding for slice {sl!r}.")

    mapped = map_sc_reference_to_unique_xenium_cells(
        sc_subset,
        sl_adata,
        embedding_key=key,
        spatial_cols=("center_x", "center_y"),
        n_matches=min(50, sc_subset.n_obs),
        reuse_penalty=9.0,
        rng=rng,
        max_reuse_per_sc=5000,
        match_cell_type=True,
        ensure_all_sc_used=True,
        sc_cell_type_col="cell_type",
        st_cell_type_col="cell_type",
    )

    mapped.obs[slice_col] = sl
    mapped.obs[donor_col] = donor

    # Use this column as tissue_col during admixture training to keep
    # spatial neighbors within slices, including slices from the same donor.
    mapped.obs["harmony_spatial_tissue"] = sl

    mapping_metadata[sl] = dict(mapped.uns["sc_reference_to_spatial_mapping"])
    mapped_slices.append(mapped)
    slice_names.append(sl)

sc_mapped = ad.concat(
    mapped_slices,
    axis=0,
    join="inner",
    merge="same",
    keys=slice_names,
    index_unique="::slice=",
)
sc_mapped.uns["mapping_by_slice"] = mapping_metadata

# source_sc_id identifies the original reference cell.
# matched_st_id preserves the original spatial cell ID.
# Integer match indices refer to the per-slice input subsets.

/home/jwinchel/miniconda3/envs/rctd/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/jwinchel/Syndri/code/mapping.py:637: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  mapped_sc_data.obs[f"{prefix}_{j+1}_dist"] = match_result["dists"][:, j]
/home/jwinchel/Syndri/code/mapping.py:638: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  mapped_sc_data.obs[f"{prefix}_{j+1}_index"] = match

In [15]:
sc_mapped.obs_names_make_unique()

In [16]:
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold

# Make mapping metadata compatible with H5AD serialization.
if "sc_reference_to_spatial_mapping" in sc_mapped.uns:
    metadata = sc_mapped.uns["sc_reference_to_spatial_mapping"]
    metadata["spatial_cols"] = list(metadata["spatial_cols"])

for metadata in sc_mapped.uns.get("mapping_by_slice", {}).values():
    metadata["spatial_cols"] = list(metadata["spatial_cols"])

if sc_mapped.obs[["cond_label", "source_sc_id"]].isna().any().any():
    raise ValueError("cond_label and source_sc_id must not contain missing values.")

splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=0,
)

_, val_idx = next(
    splitter.split(
        X=np.zeros((sc_mapped.n_obs, 1)),
        y=sc_mapped.obs["cond_label"].astype(str).to_numpy(),
        groups=sc_mapped.obs["source_sc_id"].astype(str).to_numpy(),
    )
)

split_labels = np.full(sc_mapped.n_obs, "train", dtype=object)
split_labels[val_idx] = "val"
sc_mapped.obs["split"] = split_labels

print(
    sc_mapped.obs.groupby(["cond_label", "split"], observed=True)
    .size()
    .unstack(fill_value=0)
)

# Cell-type encoding.
cell_types = sorted(sc_mapped.obs["cell_type"].unique())
sc_mapped.uns["cell_type_dict"] = {
    cell_type: -(i + 1)
    for i, cell_type in enumerate(cell_types)
}

sc_mapped.write_h5ad(
    "/work/magroup/skrieger/Syndri/data/datasets/agingbrain_merfish/"
    "interim/single-cell/sc_integrated_mapped-donors5to8.h5ad"
)

/tmp/ipykernel_935187/3924047661.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  sc_mapped.obs["split"] = split_labels


split                         train   val
cond_label                               
MsBrainAgingSpatialDonor_5_0   8381  2363
MsBrainAgingSpatialDonor_5_1   7710  1934
MsBrainAgingSpatialDonor_5_2   6929  2054
MsBrainAgingSpatialDonor_6_0   7634  1828
MsBrainAgingSpatialDonor_6_1   6566  1598
MsBrainAgingSpatialDonor_6_2   6684  1597
MsBrainAgingSpatialDonor_7_0   8771  2014
MsBrainAgingSpatialDonor_7_1   8499  2110
MsBrainAgingSpatialDonor_7_2   9184  2214
MsBrainAgingSpatialDonor_8_0  11438  2684
MsBrainAgingSpatialDonor_8_1   9779  2401
MsBrainAgingSpatialDonor_8_2   9593  2443
